# T4: DuckDB benchmark (local)

Benchmarks DuckDB read/write speed and file size on the Green taxi 2024 partition (617,885 rows).  
Runs locally because DuckDB cannot be installed in Snowflake's Snowpark environment.  
Data source: `green2024.csv` exported from Snowflake Silver layer.

In [1]:
import time
import os
import numpy as np
import pandas as pd
import duckdb

print(f"duckdb {duckdb.__version__}")
print(f"pandas {pd.__version__}")

duckdb 1.5.4
pandas 3.0.3


In [2]:
CSV_PATH = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'green2024.csv')

print("Loading CSV...")
df = pd.read_csv(CSV_PATH)
print(f"Loaded: {len(df):,} rows, {len(df.columns)} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head(2)

Loading CSV...
Loaded: 617,885 rows, 24 columns
Memory: 267.1 MB


,VENDOR_ID,PICKUP_DATETIME,DROPOFF_DATETIME,STORE_AND_FWD_FLAG,RATECODE_ID,PU_LOCATION_ID,DO_LOCATION_ID,PASSENGER_COUNT,TRIP_DISTANCE,FARE_AMOUNT,...,EHAIL_FEE,IMPROVEMENT_SURCHARGE,TOTAL_AMOUNT,PAYMENT_TYPE,TRIP_TYPE,CONGESTION_SURCHARGE,CBD_CONGESTION_FEE,PICKUP_YEAR,PICKUP_MONTH,SOURCE_FILE
0,1,2024-12-23 14:09:49.000,2024-12-23 14:26:16.000,N,1.0,18,32,1.0,2.5,17.0,...,NaN,1.0,19.5,2.0,1.0,0.0,NaN,2024,12,green_tripdata_2024-12.parquet
1,1,2024-12-23 14:35:09.000,2024-12-23 14:35:23.000,N,1.0,243,243,1.0,7.2,3.0,...,NaN,1.0,4.5,4.0,1.0,0.0,NaN,2024,12,green_tripdata_2024-12.parquet


In [3]:
OUT_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'tmp_t4')
os.makedirs(OUT_DIR, exist_ok=True)

DB_PATH = os.path.join(OUT_DIR, 'green_2024.duckdb')
results = []

def measure_read(read_fn, n=3):
    times = []
    for _ in range(n):
        start = time.perf_counter()
        read_fn()
        times.append(time.perf_counter() - start)
    return float(np.median(times))

# ── Write ────────────────────────────────────────────────────────────────────
# Remove stale file so we measure a cold write every time
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

start = time.perf_counter()
con = duckdb.connect(DB_PATH)
con.execute("CREATE TABLE green_2024 AS SELECT * FROM df")
con.close()
write_time = time.perf_counter() - start

size_mb = os.path.getsize(DB_PATH) / 1e6
print(f"Write: {write_time:.3f}s  |  File size: {size_mb:.1f} MB")

# ── Read (full scan, median of 3) ────────────────────────────────────────────
def read_duckdb():
    con = duckdb.connect(DB_PATH, read_only=True)
    result = con.execute("SELECT * FROM green_2024").df()
    con.close()
    return result

read_time = measure_read(read_duckdb)
print(f"Read (median of 3): {read_time:.3f}s")

results.append({
    'FORMAT': 'DuckDB',
    'FILE_SIZE_MB': round(size_mb, 1),
    'READ_TIME_SEC': round(read_time, 3),
    'WRITE_TIME_SEC': round(write_time, 3),
    'ENGINE': 'Local/DuckDB'
})

print("\nDone.")

Write: 1.756s  |  File size: 23.6 MB
Read (median of 3): 0.618s

Done.


In [4]:
# ── Summary table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Save as CSV so the numbers can be copied into the Snowflake T4 notebook
out_csv = os.path.join(OUT_DIR, 't4_duckdb_results.csv')
results_df.to_csv(out_csv, index=False)
print(f"\nSaved to {out_csv}")

FORMAT  FILE_SIZE_MB  READ_TIME_SEC  WRITE_TIME_SEC       ENGINE
DuckDB          23.6          0.618           1.756 Local/DuckDB

Saved to C:\Users\MatijaBazec\Documents\Faks\bd_project\Project\scripts\t3\tmp_t4\t4_duckdb_results.csv


In [5]:
# ── Quick schema / sanity check ───────────────────────────────────────────────
con = duckdb.connect(DB_PATH, read_only=True)
print("Row count:", con.execute("SELECT COUNT(*) FROM green_2024").fetchone()[0])
print("\nSchema:")
print(con.execute("DESCRIBE green_2024").df().to_string(index=False))
con.close()

Row count: 617885

Schema:
          column_name column_type null  key default extra
            VENDOR_ID      BIGINT  YES None    None  None
      PICKUP_DATETIME     VARCHAR  YES None    None  None
     DROPOFF_DATETIME     VARCHAR  YES None    None  None
   STORE_AND_FWD_FLAG     VARCHAR  YES None    None  None
          RATECODE_ID      DOUBLE  YES None    None  None
       PU_LOCATION_ID      BIGINT  YES None    None  None
       DO_LOCATION_ID      BIGINT  YES None    None  None
      PASSENGER_COUNT      DOUBLE  YES None    None  None
        TRIP_DISTANCE      DOUBLE  YES None    None  None
          FARE_AMOUNT      DOUBLE  YES None    None  None
                EXTRA      DOUBLE  YES None    None  None
              MTA_TAX      DOUBLE  YES None    None  None
           TIP_AMOUNT      DOUBLE  YES None    None  None
         TOLLS_AMOUNT      DOUBLE  YES None    None  None
            EHAIL_FEE      DOUBLE  YES None    None  None
IMPROVEMENT_SURCHARGE      DOUBLE  YES None  